# MobileNetV2-CIFAR10-Compression — Google Colab GPU Runner

This notebook runs the complete pipeline end-to-end on a Google Colab GPU (T4 / L4 / A100):
1. Mount Google Drive for persistent storage (checkpoints & results survive Colab disconnects).
2. Clone/sync your GitHub repository (or load from Google Drive) and install dependencies.
3. Run the fast smoke test to verify custom BN folding, structured pruning, STE quantization, and live bit-packing.
4. Train the FP32 MobileNet-v2 baseline on CIFAR-10 (~10–12 min on T4; target 93–94%+ Top-1 accuracy).
5. Evaluate baseline failure modes (loss/accuracy curves, confusion matrix, misclassified samples).
6. Run a single CLI verification (`python test.py --weight_quant_bits 8 --activation_quant_bits 8`).
7. Run the resume-safe 16-configuration compression sweep logged to **Weights & Biases (WandB)** for the Parallel Coordinates chart.
8. Print the final summary table & checklist numbers for your submission PDF.

In [ ]:
# 1. Check GPU & Mount Google Drive
!nvidia-smi
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Clone GitHub repository & set up Google Drive persistence
import os

REPO_URL = "https://github.com/Rakesh4ever/MobileNetV2-CIFAR10-Compression.git"
PROJECT_DIR = "/content/MobileNetV2-CIFAR10-Compression"
DRIVE_WORK_DIR = "/content/drive/MyDrive/MobileNetV2-CIFAR10-Compression"
os.makedirs(DRIVE_WORK_DIR, exist_ok=True)

if not os.path.exists(PROJECT_DIR):
    !git clone {REPO_URL} {PROJECT_DIR}
else:
    !git -C {PROJECT_DIR} pull

%cd {PROJECT_DIR}
os.environ["PYTHONPATH"] = PROJECT_DIR
assert os.path.isdir(os.path.join(PROJECT_DIR, "src")), (
    f"'src/' directory not found inside {PROJECT_DIR}. Verify the GitHub repo URL and branch."
)
print(f"Working directory verified: {os.getcwd()}")
!ls -la

In [ ]:
# 3. Install pinned dependencies & login to Weights & Biases
!pip install -q wandb seaborn scikit-learn tqdm pyyaml
import wandb
wandb.login()  # Paste your WandB API key when prompted

In [ ]:
# 4. Run smoke test (<10s) to confirm all custom modules pass on Colab GPU
os.chdir(PROJECT_DIR)
!PYTHONPATH="{PROJECT_DIR}" python scripts/smoke_test.py

In [ ]:
# 5. Train MobileNet-v2 FP32 Baseline on CIFAR-10 (saves checkpoints & results to Drive)
os.chdir(PROJECT_DIR)
CKPT_DIR = f"{DRIVE_WORK_DIR}/checkpoints"
RES_DIR = f"{DRIVE_WORK_DIR}/results"
WANDB_PROJECT = "mobilenetv2-cifar10-compression"

if not os.path.exists(f"{CKPT_DIR}/baseline_best.pth"):
    !PYTHONPATH="{PROJECT_DIR}" python -m src.train \
        --epochs 200 \
        --batch_size 128 \
        --lr 0.1 \
        --weight_decay 5e-4 \
        --warmup_epochs 5 \
        --label_smoothing 0.05 \
        --seed 42 \
        --save_dir "{CKPT_DIR}" \
        --results_dir "{RES_DIR}" \
        --wandb \
        --wandb_project "{WANDB_PROJECT}" \
        --no_tqdm
    assert os.path.exists(f"{CKPT_DIR}/baseline_best.pth"), "Baseline training failed — check logs above!"
else:
    print("Baseline checkpoint already exists on Google Drive; skipping training!")

In [ ]:
# 6. Display Baseline Curves, Confusion Matrix, and Failure Modes (Question 1c)
from IPython.display import Image, display
import json
import os

for fname in ["baseline_curves.png", "confusion_matrix.png", "failure_modes.png"]:
    p = f"{RES_DIR}/{fname}"
    if os.path.exists(p):
        display(Image(filename=p))

eval_path = f"{RES_DIR}/baseline_eval.json"
if os.path.exists(eval_path):
    with open(eval_path) as f:
        eval_data = json.load(f)
    print(json.dumps(eval_data, indent=2))
else:
    print(f"Warning: '{eval_path}' not found. Make sure Step 5 completed without errors.")

In [ ]:
# 7. Verify template CLI command (W8A8) & extended CLI command (Prune 40% + W4A4 QAT)
os.chdir(PROJECT_DIR)
!PYTHONPATH="{PROJECT_DIR}" python test.py --weight_quant_bits 8 --activation_quant_bits 8 --qat_epochs 5 --checkpoint "{CKPT_DIR}/baseline_best.pth" --no_tqdm
!PYTHONPATH="{PROJECT_DIR}" python test.py --weight_quant_bits 4 --activation_quant_bits 4 --prune_ratio 0.40 --qat_epochs 20 --checkpoint "{CKPT_DIR}/baseline_best.pth" --save_json "{RES_DIR}/sample_w4a4_p40.json" --no_tqdm

In [ ]:
# 8. Run Full 16-Configuration Compression Sweep & Log to WandB (Resume-Safe!)
# Tip: Use --quick for 8 configs (~25-35 min on T4) or omit --quick for all 16 configs (~60-75 min on T4).
os.chdir(PROJECT_DIR)
!PYTHONPATH="{PROJECT_DIR}" python sweep.py \
    --checkpoint "{CKPT_DIR}/baseline_best.pth" \
    --results_dir "{RES_DIR}" \
    --resume \
    --wandb \
    --wandb_project "{WANDB_PROJECT}" \
    --no_tqdm

In [ ]:
# 9. Display Sweep Summary Table & Plots (for Questions 3 & 4)
import pandas as pd
from IPython.display import Image, display

df = pd.read_csv(f"{RES_DIR}/sweep_results.csv")
display(df.sort_values(by=["quantized_acc", "compression_ratio"], ascending=[False, False]))

for fname in ["pareto_frontier.png", "parallel_coordinates_local.png"]:
    p = f"{RES_DIR}/{fname}"
    if os.path.exists(p):
        display(Image(filename=p))

print("\nNext Step: Open your WandB project -> Add Panel -> Parallel Coordinates")
print("Select axes: activation_quant_bits, weight_quant_bits, prune_ratio, compression_ratio, model_size_mb, quantized_acc")
print("Export that WandB Parallel Coordinates chart as PNG/SVG and insert it into your final PDF!")